[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees/cours/seance1_cours.ipynb)

# Séance 2.1 — Charger et comprendre un jeu de données

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- charger un fichier de données depuis le web en une ligne
- dire en 30 secondes ce que contient un fichier que vous n'avez jamais vu
- sélectionner les lignes et les colonnes qui vous intéressent
- calculer des indicateurs simples (moyenne, total, répartition)
- lire un message d'erreur au lieu de le subir

## Le contexte

Vous venez d'arriver chez un **détaillant en ligne** européen. On vous remet
l'historique des ventes de l'année écoulée et une question simple :

> *« Sur quel marché faut-il investir l'an prochain ? »*

Vous ne pouvez pas répondre tant que vous ne savez pas ce que contient ce
fichier. Cette séance, c'est exactement ça : **prendre en main un jeu de
données qu'on n'a jamais vu**.

Nous avons trois fichiers :

| Fichier | Une ligne = | Colonnes |
|---|---|---|
| `ventes.csv` | un produit dans une commande | `date`, `cmd_id`, `prod_id`, `qte`, `prix`, `client_id` |
| `clients.csv` | un client | `client_id`, `pays`, `segment`, `date_insc` |
| `produits.csv` | un produit | `prod_id`, `libelle`, `categorie` |

## 1. Le point de départ

Cette cellule est présente au début de **tous** les notebooks du cours. Elle
charge les outils dont nous aurons besoin et règle l'affichage pour les petits
écrans. Exécutez-la (bouton ▶) sans chercher à la comprendre pour l'instant.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

**`pandas`** est la bibliothèque qui sert à manipuler des tableaux de données
en Python. `pd` est son surnom : on écrira `pd.` chaque fois qu'on l'appelle.

Pensez à pandas comme à **un Excel qu'on pilote par des instructions**. Même
objet — un tableau de lignes et de colonnes — mais au lieu de cliquer, on écrit
ce qu'on veut. L'avantage : ça marche sur 45 000 lignes aussi vite que sur 10,
et on peut relancer exactement la même analyse le mois prochain.

## 2. Charger les données

Une seule ligne. `pd.read_csv()` accepte directement une **adresse web** :
rien à télécharger, rien à ranger dans un dossier.

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")  ## lit le fichier en ligne
ventes.head(3)                             ## les 3 premieres lignes

`ventes` est un **DataFrame** : le nom que pandas donne à un tableau.

`.head(3)` affiche les 3 premières lignes. Toujours commencer par là : c'est
la façon la plus rapide de vérifier que le fichier est bien celui qu'on croit.

> 💡 On affiche `head(3)` et pas `head()` (qui en montre 5) parce que sur un
> écran de tablette, 3 lignes tiennent et 5 obligent à faire défiler.

## 3. La carte d'identité d'un fichier

Quatre commandes, toujours dans cet ordre. En 30 secondes vous savez à quoi
vous avez affaire.

In [ ]:
ventes.shape   ## (nombre de lignes, nombre de colonnes)

In [ ]:
ventes.info()  ## les colonnes, leur type, les valeurs manquantes

Ce que `info()` vous dit, ligne par ligne :

- **`45123 entries`** — 45 123 lignes.
- **`non-null`** — combien de valeurs sont renseignées. Ici tout est complet ;
  on verra en séance 2.2 que c'est très rare dans la vraie vie.
- **`Dtype`** — le *type* de chaque colonne, et c'est le plus important :
  - `int64` : nombre entier (`qte`, `client_id`)
  - `float64` : nombre à virgule (`prix`)
  - `object` : **du texte** (`date`, `prod_id`)

> ⚠️ Regardez `date` : son type est `object`, c'est-à-dire du **texte**. Pour
> pandas, `"2011-10-04"` est une chaîne de caractères, pas une date. On ne peut
> donc pas encore lui demander « quel mois ? ». On corrigera ça en séance 2.2.

In [ ]:
# 45 123 lignes, mais combien de clients et de commandes distincts ?
print("clients   :", ventes["client_id"].nunique())   ## valeurs distinctes
print("commandes :", ventes["cmd_id"].nunique())      ## idem sur la commande

**Une ligne n'est pas un client.** Une ligne est *un produit dans une
commande*. 45 123 lignes correspondent à 1 955 commandes passées par 472
clients.

C'est la première question à se poser devant n'importe quel fichier :
**une ligne, c'est quoi exactement ?** Se tromper là-dessus, c'est se tromper
sur tout le reste de l'analyse.

## 4. Choisir ce qu'on regarde

### Une colonne

In [ ]:
ventes["prix"].head(3)   ## une seule colonne : c'est une Series

### Plusieurs colonnes

Doubles crochets : les crochets extérieurs veulent dire « je sélectionne »,
les intérieurs délimitent la **liste** des colonnes voulues.

In [ ]:
ventes[["prod_id", "qte", "prix"]].head(3)  ## une liste -> un tableau

### Une ligne par sa position — `.iloc`

In [ ]:
ventes.iloc[0]     ## la toute premiere ligne, par sa position

### Des lignes par une condition — `.query()`

C'est ici que ça devient utile. On veut les ventes dont le prix dépasse 50 € :

In [ ]:
cheres = ventes.query("prix > 50")   ## la condition, entre guillemets
print(cheres.shape)                  ## combien de lignes ont survecu ?
cheres.head(3)

`.query()` prend une **condition écrite entre guillemets**, presque en français :
`"prix > 50"`, `"qte >= 100"`, `"pays == 'France'"`.

Vous rencontrerez aussi l'autre écriture, plus classique :

```python
ventes[ventes["prix"] > 50]
```

Les deux font exactement la même chose. **Nous utiliserons `.query()` dans ce
cours** : deux fois moins de ponctuation à taper, et beaucoup plus lisible dès
que la condition se complique.

## 5. Résumer en un coup d'œil

### `describe()` — le résumé chiffré

In [ ]:
# On selectionne les colonnes AVANT : sinon la sortie deborde de l'ecran
ventes[["qte", "prix"]].describe().round(2)   ## huit statistiques d'un coup

À lire ainsi :

- `mean` : la moyenne. Prix moyen : **3,93 €**.
- `50%` : la **médiane**, la valeur qui coupe la population en deux. **1,95 €**.
- `max` : la valeur maximale. **4 161 €**.

> 📊 Moyenne 3,93 € mais médiane 1,95 € : la moyenne est **deux fois** la
> médiane. C'est la signature d'une poignée de valeurs très élevées qui tirent
> la moyenne vers le haut. Devant un écart pareil, la médiane décrit bien mieux
> « le produit typique ». Un réflexe à garder : **comparer moyenne et médiane
> avant de citer un chiffre en réunion.**

### `value_counts()` — compter les catégories

In [ ]:
clients = pd.read_csv(BASE + "clients.csv")
clients["pays"].value_counts().head(5)   ## deja trie du plus frequent

### Un seul chiffre à la fois

In [ ]:
grosses = ventes.query("qte > 100")   ## un sous-tableau, filtre

print("prix moyen :", ventes["prix"].mean().round(2))   ## .round() arrondit
print("quantite max :", ventes["qte"].max())            ## le maximum
print("lignes a plus de 100 unites :", grosses.shape[0])   ## shape[0] = lignes

## 6. Créer une colonne

Le chiffre d'affaires d'une ligne, c'est la quantité multipliée par le prix.

In [ ]:
ventes["ca"] = ventes["qte"] * ventes["prix"]  ## 45 123 calculs
ventes[["qte", "prix", "ca"]].head(3)          ## toujours verifier apres

Regardez bien ce qui vient de se passer. Nous n'avons **pas** écrit de boucle.
Une seule instruction a fait 45 123 multiplications.

C'est ce qu'on appelle la **vectorisation** : l'opération est appliquée d'un
coup à toute la colonne. C'est **numpy** (`np`) qui travaille en dessous, et
c'est des dizaines de fois plus rapide qu'une boucle Python.

La règle à retenir : **si vous vous surprenez à écrire une boucle sur les
lignes d'un DataFrame, il existe presque toujours une façon vectorisée de le
faire.**

In [ ]:
# Le chiffre d'affaires total de l'annee
print("CA total :", round(ventes["ca"].sum(), 2), "euros")   ## somme colonne

## 7. Apprendre à lire une erreur

Vous allez faire des erreurs en permanence. C'est normal, y compris pour les
professionnels. Ce qui distingue quelqu'un qui avance, c'est qu'il **lit** le
message au lieu de le subir.

La cellule suivante est **volontairement fausse**. Exécutez-la.

In [ ]:
ventes["Prix"]   ## erreur volontaire : la colonne est "prix"

Le message est long et rouge. **Ne lisez que la dernière ligne :**

```
KeyError: 'Prix'
```

Traduction : *« je n'ai trouvé aucune colonne appelée `Prix` »*. Python
distingue majuscules et minuscules : `Prix` et `prix` sont deux noms
différents.

### Les trois erreurs que vous verrez le plus souvent

| Message | Ce que ça veut dire | Quoi faire |
|---|---|---|
| `KeyError: 'Prix'` | cette colonne n'existe pas | vérifier l'orthographe avec `ventes.columns` |
| `NameError: name 'vente' is not defined` | cette variable n'existe pas | faute de frappe, ou cellule au-dessus pas exécutée |
| `SyntaxError: invalid character '"'` | guillemets « intelligents » | 📱 désactiver la Ponctuation intelligente (voir *Bien démarrer*) |

> 📱 La troisième est spécifique aux tablettes et **invisible à l'œil nu** :
> votre code paraît parfaitement correct. Si vous ne l'avez pas encore fait,
> faites le réglage maintenant.

In [ ]:
# Le reflexe quand on ne se souvient plus d'un nom de colonne
ventes.columns   ## la liste exacte, majuscules comprises

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| charger un fichier | `pd.read_csv(url)` |
| voir les premières lignes | `df.head(3)` |
| connaître la taille | `df.shape` |
| voir les colonnes et leurs types | `df.info()` |
| une colonne | `df["prix"]` |
| plusieurs colonnes | `df[["prix", "qte"]]` |
| une ligne par sa position | `df.iloc[0]` |
| des lignes par condition | `df.query("prix > 10")` |
| résumé chiffré | `df.describe()` |
| compter les catégories | `df["pays"].value_counts()` |
| moyenne, total, max | `df["prix"].mean()`, `.sum()`, `.max()` |
| créer une colonne | `df["ca"] = df["qte"] * df["prix"]` |

**Le réflexe à garder :** devant un fichier inconnu, toujours dans cet ordre —
`shape`, `info()`, `head(3)`, `describe()`. Quatre commandes, et vous savez à
quoi vous avez affaire.